# 소재 온톨로지 실습

**Materials Ontology · EMMO**

소재 분야의 물질·구조·공정·물성 개념을 표준화해 정의한 온톨로지. 데이터 통합과 자동 해석의 기준이 된다.

소재 분야에서 이해하기: 같은 물성을 서로 다른 이름으로 기록한 데이터셋을 표준 개념으로 정렬한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [EMMO 소재 분야 온톨로지](https://emmo-repo.github.io/)

## 1. 같은 물성, 다른 기록

두 실험실이 같은 것을 다른 이름과 단위로 기록한 상황을 만들어 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import pandas as pd

lab_a = pd.DataFrame([
    {'sample': 'A1', 'hardness_HV': 431, 'sinter_temp_C': 780, 'hold_h': 4.0},
    {'sample': 'A2', 'hardness_HV': 455, 'sinter_temp_C': 860, 'hold_h': 2.0},
])
lab_b = pd.DataFrame([
    {'specimen': 'B1', 'HV0.5': 388, 'T_sinter[K]': 1053, 'dwell_min': 240},
    {'specimen': 'B2', 'HV0.5': 402, 'T_sinter[K]': 1133, 'dwell_min': 120},
])
print(lab_a); print(); print(lab_b)
print('\n열 이름도 단위도 다릅니다. 그대로 이어붙이면 같은 축에 올릴 수 없습니다.')

## 2. 개념으로 정렬하기

표준 개념과 단위를 정해두고, 각 데이터셋의 열을 그 개념에 연결합니다.

In [ ]:
# 표준 개념: 정식 이름과 기준 단위
CONCEPTS = {
    'VickersHardness': 'HV',
    'SinteringTemperature': 'C',
    'HoldTime': 'h',
    'SampleIdentifier': None,
}

# 각 데이터셋의 열 -> (개념, 변환 함수)
MAPPING_A = {'sample': ('SampleIdentifier', lambda v: v),
             'hardness_HV': ('VickersHardness', lambda v: v),
             'sinter_temp_C': ('SinteringTemperature', lambda v: v),
             'hold_h': ('HoldTime', lambda v: v)}
MAPPING_B = {'specimen': ('SampleIdentifier', lambda v: v),
             'HV0.5': ('VickersHardness', lambda v: v),
             'T_sinter[K]': ('SinteringTemperature', lambda v: v - 273.15),
             'dwell_min': ('HoldTime', lambda v: v / 60.0)}

def align(frame, mapping):
    rows = []
    for record in frame.to_dict('records'):
        aligned = {}
        for column, value in record.items():
            concept, convert = mapping[column]
            aligned[concept] = convert(value)
        rows.append(aligned)
    return pd.DataFrame(rows)[list(CONCEPTS)]

merged = pd.concat([align(lab_a, MAPPING_A), align(lab_b, MAPPING_B)], ignore_index=True)
print(merged.round(2))
print('\n기준 단위:', {k: v for k, v in CONCEPTS.items() if v})

## 3. 정렬하지 않으면 무슨 일이 생기나

In [ ]:
naive = pd.concat([
    lab_a.rename(columns={'sample': 'id', 'hardness_HV': 'hardness', 'sinter_temp_C': 'temperature'}),
    lab_b.rename(columns={'specimen': 'id', 'HV0.5': 'hardness', 'T_sinter[K]': 'temperature'}),
], ignore_index=True)
print('개념 정렬 없이 합친 온도 열:')
print(naive[['id', 'temperature']].to_string(index=False))
print('\n평균 온도 %.1f — 섭씨와 켈빈이 섞여 아무 의미가 없습니다.' % naive['temperature'].mean())
print('정렬 후 평균 온도 %.1f C' % merged['SinteringTemperature'].mean())

## 4. 개념 계층으로 넓게 묻기

In [ ]:
PROPERTY_TREE = {'VickersHardness': 'Hardness', 'Hardness': 'MechanicalProperty',
                 'BandGap': 'ElectronicProperty'}

def under(concept, target):
    while concept:
        if concept == target:
            return True
        concept = PROPERTY_TREE.get(concept)
    return False

print('MechanicalProperty 에 속하는 개념:',
      [c for c in list(PROPERTY_TREE) + ['MechanicalProperty'] if under(c, 'MechanicalProperty')])
print('\n표준 온톨로지를 쓰면 이 계층을 직접 정의하지 않고 재사용할 수 있습니다.')
print('실무에서는 EMMO 같은 소재 온톨로지를 기반으로 부족한 개념만 확장합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#materials-ontology)을 여세요.